# 🔐 Day 01a: JWT Authentication — Stateless Auth for APIs

---

## 🎯 What You'll Master Today
- JWT structure: header.payload.signature
- Access tokens vs refresh tokens
- Password hashing with bcrypt
- Complete JWT auth system in FastAPI
- JWT vs session-based auth

---

## 🎭 How JWT Works

```
╔═══════════════════════════════════════════════════════════════════╗
║  JWT = JSON WEB TOKEN                                            ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Structure: HEADER.PAYLOAD.SIGNATURE                             ║
║                                                                   ║
║  ┌─────────────────────────────────────────────┐                  ║
║  │ HEADER:  {"alg": "HS256", "typ": "JWT"}    │ ← Algorithm      ║
║  ├─────────────────────────────────────────────┤                  ║
║  │ PAYLOAD: {"sub": "user_id", "exp": 16...}  │ ← Claims/Data   ║
║  ├─────────────────────────────────────────────┤                  ║
║  │ SIGNATURE: HMAC(header+payload, SECRET_KEY) │ ← Verification  ║
║  └─────────────────────────────────────────────┘                  ║
║                                                                   ║
║  Flow:                                                            ║
║  1. Client sends username + password                              ║
║  2. Server verifies, creates JWT, sends back                     ║
║  3. Client stores JWT, sends in Authorization header             ║
║  4. Server verifies JWT signature on each request                ║
║  5. No database lookup needed! (stateless)                       ║
║                                                                   ║
║  Header format:  Authorization: Bearer <token>                   ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# Password Hashing — NEVER store plain text passwords!
# ============================================================
# pip install passlib[bcrypt] python-jose[cryptography]

from passlib.context import CryptContext

# bcrypt context
pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")

def hash_password(password: str) -> str:
    """Hash a password for storage."""
    return pwd_context.hash(password)

def verify_password(plain_password: str, hashed_password: str) -> bool:
    """Verify a password against its hash."""
    return pwd_context.verify(plain_password, hashed_password)

# Demo
password = "MySecure123!"
hashed = hash_password(password)
print(f"Original: {password}")
print(f"Hashed:   {hashed}")
print(f"Verify correct:  {verify_password(password, hashed)}")
print(f"Verify wrong:    {verify_password('WrongPass', hashed)}")
print(f"\n🔑 bcrypt auto-includes salt — same password gives different hashes!")
print(f"Hash 2:   {hash_password(password)}")

In [ ]:
# ============================================================
# JWT Token Creation & Verification
# ============================================================
from jose import JWTError, jwt
from datetime import datetime, timedelta

# Config — in production, use environment variables!
SECRET_KEY = "your-secret-key-change-in-production"  # use: openssl rand -hex 32
ALGORITHM = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30
REFRESH_TOKEN_EXPIRE_DAYS = 7

def create_access_token(data: dict, expires_delta: timedelta = None) -> str:
    """Create a JWT access token."""
    to_encode = data.copy()
    expire = datetime.utcnow() + (expires_delta or timedelta(minutes=ACCESS_TOKEN_EXPIRE_MINUTES))
    to_encode.update({"exp": expire, "type": "access"})
    return jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)

def create_refresh_token(data: dict) -> str:
    """Create a JWT refresh token (longer-lived)."""
    to_encode = data.copy()
    expire = datetime.utcnow() + timedelta(days=REFRESH_TOKEN_EXPIRE_DAYS)
    to_encode.update({"exp": expire, "type": "refresh"})
    return jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)

def decode_token(token: str) -> dict:
    """Decode and verify a JWT token."""
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        return payload
    except JWTError as e:
        raise ValueError(f"Invalid token: {e}")

# Demo
access = create_access_token({"sub": "user_123", "role": "admin"})
refresh = create_refresh_token({"sub": "user_123"})

print(f"Access token:  {access[:50]}...")
print(f"Refresh token: {refresh[:50]}...")

# Decode
payload = decode_token(access)
print(f"\nDecoded payload: {payload}")
print(f"User ID: {payload['sub']}")
print(f"Token type: {payload['type']}")

In [ ]:
# ============================================================
# Complete FastAPI JWT Auth System
# ============================================================
from fastapi import FastAPI, Depends, HTTPException, status
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from fastapi.testclient import TestClient
from pydantic import BaseModel
from typing import Optional

app = FastAPI()

# OAuth2 scheme — tells FastAPI where to find the token
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="/auth/login")

# Fake DB
fake_users_db = {}

# Schemas
class UserCreate(BaseModel):
    username: str
    password: str

class TokenResponse(BaseModel):
    access_token: str
    refresh_token: str
    token_type: str = "bearer"

class UserResponse(BaseModel):
    username: str
    role: str

# Auth dependency — extracts and verifies the current user
async def get_current_user(token: str = Depends(oauth2_scheme)):
    """Dependency: extracts user from JWT token."""
    credentials_exception = HTTPException(
        status_code=status.HTTP_401_UNAUTHORIZED,
        detail="Invalid or expired token",
        headers={"WWW-Authenticate": "Bearer"},
    )
    try:
        payload = decode_token(token)
        username = payload.get("sub")
        if username is None or payload.get("type") != "access":
            raise credentials_exception
    except ValueError:
        raise credentials_exception
    
    user = fake_users_db.get(username)
    if user is None:
        raise credentials_exception
    return user

# Routes
@app.post("/auth/register")
def register(user: UserCreate):
    if user.username in fake_users_db:
        raise HTTPException(400, "Username taken")
    fake_users_db[user.username] = {
        "username": user.username,
        "password_hash": hash_password(user.password),
        "role": "user"
    }
    return {"message": "User created"}

@app.post("/auth/login", response_model=TokenResponse)
def login(form: OAuth2PasswordRequestForm = Depends()):
    user = fake_users_db.get(form.username)
    if not user or not verify_password(form.password, user["password_hash"]):
        raise HTTPException(401, "Invalid credentials")
    
    access = create_access_token({"sub": user["username"], "role": user["role"]})
    refresh = create_refresh_token({"sub": user["username"]})
    return TokenResponse(access_token=access, refresh_token=refresh)

@app.get("/users/me", response_model=UserResponse)
def get_me(current_user: dict = Depends(get_current_user)):
    return UserResponse(username=current_user["username"], role=current_user["role"])

# Test it!
client = TestClient(app)

# Register
r = client.post("/auth/register", json={"username": "sarthak", "password": "Secure123!"})
print(f"Register: {r.json()}")

# Login
r = client.post("/auth/login", data={"username": "sarthak", "password": "Secure123!"})
tokens = r.json()
print(f"Login: tokens received ✅")

# Access protected route
r = client.get("/users/me", headers={"Authorization": f"Bearer {tokens['access_token']}"})
print(f"Profile: {r.json()}")

# Try without token
r = client.get("/users/me")
print(f"No token: {r.status_code} — {r.json()['detail']}")

In [ ]:
# ============================================================
# Token Refresh Pattern
# ============================================================

@app.post("/auth/refresh", response_model=TokenResponse)
def refresh_token(refresh_token_str: str):
    """Issue new access token using refresh token."""
    try:
        payload = decode_token(refresh_token_str)
        if payload.get("type") != "refresh":
            raise HTTPException(401, "Invalid token type")
        
        username = payload.get("sub")
        user = fake_users_db.get(username)
        if not user:
            raise HTTPException(401, "User not found")
        
        new_access = create_access_token({"sub": username, "role": user["role"]})
        new_refresh = create_refresh_token({"sub": username})
        return TokenResponse(access_token=new_access, refresh_token=new_refresh)
    except ValueError:
        raise HTTPException(401, "Invalid refresh token")

print("""
Token Flow:

  1. Login → get access_token (30 min) + refresh_token (7 days)
  2. Use access_token for API calls
  3. When access_token expires → call /auth/refresh with refresh_token
  4. Get new access_token + refresh_token
  5. When refresh_token expires → user must login again

Why two tokens?
  • access_token: short-lived, sent with every request
  • refresh_token: long-lived, stored securely, used rarely
  • If access_token is stolen → attacker has limited time window
""")

---

## 🗺️ JWT Auth Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  JWT AUTHENTICATION PATTERNS                                     ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  HASH:     pwd_context.hash(password)                            ║
║  VERIFY:   pwd_context.verify(plain, hashed)                     ║
║                                                                   ║
║  CREATE:   jwt.encode({sub, exp, type}, SECRET, ALGORITHM)       ║
║  DECODE:   jwt.decode(token, SECRET, algorithms=[ALG])           ║
║                                                                   ║
║  FASTAPI:  oauth2_scheme = OAuth2PasswordBearer(tokenUrl=...)    ║
║  DEPEND:   token: str = Depends(oauth2_scheme)                   ║
║  PROTECT:  current_user = Depends(get_current_user)              ║
║                                                                   ║
║  ACCESS:   Short-lived (15-30 min), sent with every request      ║
║  REFRESH:  Long-lived (7-30 days), used to get new access        ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | What is JWT? | JSON Web Token — self-contained, stateless auth token (header.payload.signature) |
| 2 | JWT vs session auth? | JWT: stateless, scalable, no server storage. Session: server-side, can revoke instantly |
| 3 | How to revoke a JWT? | Can't directly. Use: short expiry + refresh tokens, or token blacklist in Redis |
| 4 | Why bcrypt? | Adaptive, auto-salts, intentionally slow (resists brute force). Cost factor adjustable |
| 5 | Access vs refresh token? | Access: short-lived, used on every request. Refresh: long-lived, used to get new access |
| 6 | Where to store tokens? | httpOnly cookie (safest) or localStorage (easier, XSS vulnerable) |
| 7 | What's OAuth2PasswordBearer? | FastAPI security scheme that extracts Bearer token from Authorization header |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • JWT = stateless auth: header.payload.signature       │
## │  • NEVER store plain text passwords — use bcrypt         │
## │  • Access token (30m) + Refresh token (7d) pattern      │
## │  • OAuth2PasswordBearer extracts token from header      │
## │  • get_current_user dependency protects endpoints       │
## │  • Token revocation: blacklist in Redis or short expiry │
## │  • Store SECRET_KEY in environment variables             │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **OAuth2 Flows** — Social login and authorization patterns